# MedTriage AI — Model Training & Exploration
End-to-end notebook for the AI Medical Symptom Triage system.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'backend'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
%matplotlib inline
plt.style.use('dark_background')

## 1. Generate & Explore Dataset

In [ ]:
from data.generate_dataset import build_dataset
df = build_dataset(n_per_disease=300)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(12, 5))
df['disease'].value_counts().plot(kind='barh', ax=ax, color='#06b6d4')
ax.set_title('Disease Class Distribution', fontsize=14)
ax.set_xlabel('Count')
plt.tight_layout()
plt.savefig('../docs/class_distribution.png', dpi=150)
plt.show()

## 2. Train Model

In [ ]:
DROP_COLS = ['disease', 'risk_level', 'risk_score']
SYMPTOM_COLS = [c for c in df.columns if c not in DROP_COLS]

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X = df[SYMPTOM_COLS].values
y = le.fit_transform(df['disease'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)} | Classes: {len(le.classes_)}')

In [ ]:
clf = RandomForestClassifier(n_estimators=300, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 3. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Confusion Matrix', fontsize=16)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../docs/confusion_matrix.png', dpi=150)
plt.show()

## 4. Feature Importance

In [ ]:
importances = pd.Series(clf.feature_importances_, index=SYMPTOM_COLS).sort_values(ascending=False)[:20]
fig, ax = plt.subplots(figsize=(10, 6))
importances.plot(kind='barh', ax=ax, color='#06b6d4')
ax.set_title('Top 20 Most Important Symptoms', fontsize=14)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../docs/feature_importance.png', dpi=150)
plt.show()

## 5. NLP Pipeline Test

In [ ]:
from utils.nlp_pipeline import extract_symptoms, build_feature_vector
from utils.triage_engine import get_triage

test_text = 'I have been experiencing fever, severe headache, joint pain, and a rash for 3 days'
feat_dict, detected = extract_symptoms(test_text, SYMPTOM_COLS)
print('Detected:', detected)

fvec = build_feature_vector(feat_dict, SYMPTOM_COLS)
proba = clf.predict_proba([fvec])[0]
top_idx = np.argsort(proba)[::-1][:5]
predictions = [
    {'disease': le.classes_[i], 'probability': float(proba[i]), 'probability_pct': round(proba[i]*100, 1),
     'risk_level': 'high'}
    for i in top_idx
]

triage = get_triage(predictions, detected, duration_days=3)
print(f"\nRisk Level  : {triage['risk_level']}")
print(f"Top Dx      : {triage['top_prediction']}")
print(f"Recommendation: {triage['recommendation'][:120]}...")